# VQ2 training on Colab

Trains the 3-DoF racing policy against the surrogate, on the **measured VQ2 course**
(`pilot/control/course/`, 17 gates) rather than the procedural generator.

Run the cells top to bottom. Cell 4 is the one you re-run with different knobs.

**Use a CPU runtime.** The environment is vectorized NumPy on CPU and dominates the step;
the network is a 2x256 MLP, so a GPU only accelerates the PPO update and does not change
the picture much. The real win here is **running several of these notebooks at once** with
different settings -- the bottleneck is hypotheses per hour, not steps per second.

## 1. Get the code

In [ ]:
REPO   = "clarity-m/vqual-2"                 #@param {type:"string"}
BRANCH = "worktree-vq2-course-training"      #@param {type:"string"}

import os, subprocess, getpass, pathlib

DEST = "/content/vqual-2"
if not os.path.exists(DEST):
    tok = getpass.getpass("GitHub token (leave blank if the repo is public): ").strip()
    url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "checkout", "-f", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(DEST)
print("cwd:", os.getcwd())
print("course package present:", os.path.exists("pilot/control/course/course_vq2.json"))

## 2. Check the environment and verify the harness

Both suites should pass before you spend GPU-hours (or CPU-hours) on a run.

In [ ]:
import sys, torch, numpy
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), "| numpy", numpy.__version__)

!python pilot/control/train/selftest.py 2>&1 | tail -3
!python pilot/control/surrogate/selfcheck.py 2>&1 | tail -2

## 3. Persist checkpoints to Drive (optional but recommended)

Colab reclaims the VM without warning. `train.py` always writes to
`pilot/control/train/checkpoints/`, so pointing that at Drive is enough -- no code change,
and `--resume` then works across a disconnect.

In [ ]:
USE_DRIVE = True                                        #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/vqual2-checkpoints" #@param {type:"string"}

import os, shutil, pathlib
CKPT = "pilot/control/train/checkpoints"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    if os.path.islink(CKPT):
        os.unlink(CKPT)
    elif os.path.isdir(CKPT):
        shutil.rmtree(CKPT)
    os.symlink(DRIVE_DIR, CKPT)
    print("checkpoints ->", os.path.realpath(CKPT))
else:
    os.makedirs(CKPT, exist_ok=True)
    print("checkpoints stay on the VM and are LOST on disconnect:", CKPT)

## 4. Train

`VQ2_FRAC` is the share of episodes flown on the measured course. `1.0` trains on VQ2
only, which is the right call when the goal is to optimize for this specific course --
it is fixed and deterministic (spec 3.5), attempts are unlimited and ranking is on time
(9.4), so overfitting to it is the point.

The one hedge worth considering is `0.9`. `course/README.md` warns that a whole leg can
rotate by 90 deg if its mod-90 quadrant came from the sketch rather than a shared
measurement -- "a discrete failure a Gaussian cannot express" -- and calls the sampled
envelope "a lower bound". A pure-VQ2 policy has memorized a track that may be wrong in a
way no amount of sampling covers; a 10% procedural share keeps gate-seeking alive for that
case. Your call.

Set `RESUME` to a checkpoint name to continue a run instead of starting fresh.

In [ ]:
NAME        = "vq2_run1"   #@param {type:"string"}
TOTAL_STEPS = 20000000     #@param {type:"integer"}
VQ2_FRAC    = 1.0          #@param {type:"slider", min:0, max:1, step:0.05}
N_ENVS      = 256          #@param {type:"integer"}
N_STEPS     = 128          #@param {type:"integer"}
FRAME_STACK = 6            #@param {type:"integer"}
SEED        = 0            #@param {type:"integer"}
DEVICE      = "cpu"        #@param ["cpu", "cuda"]
RESUME      = ""           #@param {type:"string"}
EXTRA       = ""           #@param {type:"string"}

cmd = [
    "python", "pilot/control/train/train.py",
    "--env", "surrogate",
    "--name", NAME,
    "--total-steps", str(TOTAL_STEPS),
    "--n-envs", str(N_ENVS),
    "--n-steps", str(N_STEPS),
    "--frame-stack", str(FRAME_STACK),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--checkpoint-every", "1000000",
    "--env-kwarg", f"vq2_frac={VQ2_FRAC}",
]
if RESUME:
    cmd += ["--resume", RESUME]
if EXTRA:
    cmd += EXTRA.split()

print(" ".join(cmd), "\n")

import subprocess, sys
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print("\nexit code:", p.returncode)

## 5. Reading the log

```
[ 12/610] step=393216 fps=9500 ret=-2.9 gates=1.91 coll=0.995 compl=0.00 | diff=0.00 speed_cap=0.50 ...
```

| field | meaning |
|---|---|
| `compl` | completion rate -- the number that matters. Curriculum promotes above 0.70 |
| `coll`  | collision rate. Starts near 1.0 and should fall |
| `gates` | mean gates passed before the episode ends (17 is a full VQ2 lap) |
| `ev`    | value-function explained variance. Should climb early; if it stays near 0 something is wrong |
| `diff` / `speed_cap` | the two curriculum axes, moved automatically |

Starting at `compl=0.00` with `coll` near 1.0 is expected, not a failure.

Per-update rows also land in `checkpoints/<NAME>_log.csv`.

## Running several at once

Duplicate this notebook and change `NAME` and `SEED` (and whatever you are testing).
Distinct `NAME`s never collide -- checkpoints, sidecars and logs are all name-keyed. Some
things worth varying across parallel runs:

- `VQ2_FRAC` -- 0.0 (pure procedural, the control) vs 0.8 vs 1.0
- `EXTRA = "--env-kwarg vq2_yaw_mode='bisector'"` -- fixed gate planes instead of
  randomizing across the three disagreeing yaw hypotheses
- `EXTRA = "--difficulty-start 0.3 --speed-cap-start 0.7"` -- start the curriculum higher
- `EXTRA = "--ent-coef 0.01"` -- more exploration

## Ranking checkpoints -- match the course

The eval suite defaults to PROCEDURAL courses. Ranking a VQ2-trained checkpoint against
courses it never saw selects the wrong one, silently. Always pass `--vq2-frac` matching
what you trained on:

```
!python pilot/control/evalsuite/select.py --ckpt vq2_run1_s5046272     --ckpt vq2_run1_s10092544 --baseline --vq2-frac 1.0 --seeds 0-49
```

## Known gap

Gate 9's tilt is held at vertical pending the corrected `course_vq2.json`. Once it lands,
add `--env-kwarg "vq2_tilt_deg={9:(21.0,24.0)}"` via `EXTRA`.